# Fuente 4 - API de Meteorología Histórica (Open-Meteo)

## Objetivo

Consultar datos meteorológicos históricos de la Avenida Las Palmas mediante la API de Open-Meteo, para disponer de registros horarios de precipitación, temperatura y humedad relativa que puedan relacionarse temporalmente con la información de siniestralidad.

## 1. Identificación de la fuente

**Nombre:** Historical Weather API  
**Entidad:** Open-Meteo  
**Tipo de fuente:** API REST de información meteorológica histórica  
**Formato de respuesta:** JSON  
**URL:** https://open-meteo.com/en/docs/historical-weather-api  
**Endpoint:** https://archive-api.open-meteo.com/v1/archive  
**Ubicación de referencia:** La Asomadera #2, Avenida Las Palmas, Medellín  
**Coordenadas:** latitud 6.220055, longitud -75.562449

La fuente entrega series meteorológicas horarias estimadas mediante conjuntos de datos de modelos y reanálisis. No corresponde a mediciones de una estación instalada sobre la curva.

In [ ]:
import requests
import pandas as pd
from IPython.display import display

url_api = "https://archive-api.open-meteo.com/v1/archive"

parametros = {
    "latitude": 6.220055,
    "longitude": -75.562449,
    "start_date": "2025-03-01",
    "end_date": "2025-03-31",
    "hourly": "precipitation,temperature_2m,relative_humidity_2m",
    "timezone": "America/Bogota"
}

respuesta = requests.get(url_api, params=parametros, timeout=60)
respuesta.raise_for_status()
datos = respuesta.json()

if "error" in datos or "hourly" not in datos:
    raise ValueError(f"Respuesta meteorológica no válida: {datos}")

print("Fuente:", url_api)
print("Periodo:", parametros["start_date"], "a", parametros["end_date"])
print("Zona horaria:", datos.get("timezone"))
print("Coordenadas de la celda consultada:", datos.get("latitude"), datos.get("longitude"))

## 2. Variables disponibles en la fuente

A partir de la respuesta de la API se identifican los campos horarios y sus unidades, para conocer la estructura de los registros descargados.

In [ ]:
unidades = datos.get("hourly_units", {})
variables = pd.DataFrame([
    {"nombre_variable": nombre, "unidad": unidades.get(nombre, "")}
    for nombre in datos["hourly"].keys()
])
display(variables)

### Interpretación

Las variables seleccionadas incluyen:

- **Temporal:** `time`, fecha y hora local de cada registro.
- **Precipitación:** `precipitation`, acumulado de precipitación de la hora precedente, expresado en milímetros.
- **Temperatura:** `temperature_2m`, temperatura del aire a dos metros de altura, en grados Celsius.
- **Humedad:** `relative_humidity_2m`, humedad relativa del aire, en porcentaje.

Se utiliza la zona horaria `America/Bogota` para mantener coherencia con las horas de los registros de siniestralidad.

## 3. Consulta de registros de la fuente

Se consultan los datos horarios del periodo seleccionado y se revisan algunos de los registros devueltos.

In [ ]:
registros = pd.DataFrame(datos["hourly"])

columnas_esperadas = ["time", "precipitation", "temperature_2m", "relative_humidity_2m"]
faltantes = set(columnas_esperadas) - set(registros.columns)
if faltantes:
    raise ValueError(f"Faltan variables en la respuesta: {faltantes}")

print("Cantidad de registros:", len(registros))
print("Columnas:", registros.columns.tolist())
display(registros.head(10))

## 4. Adquisición de los datos de la fuente

Los registros horarios obtenidos se conservan con los campos originales de la respuesta, sin reemplazar datos faltantes ni modificar sus valores. El archivo descargado servirá como entrada para las etapas posteriores de integración.

In [ ]:
df_meteorologia_raw = registros.copy()
print("Dimensiones de los datos adquiridos:", df_meteorologia_raw.shape)
display(df_meteorologia_raw.tail())

### Almacenamiento de la fuente adquirida

Los datos se almacenan en CSV, utilizando el mismo separador y codificación definidos para las otras fuentes del proyecto.

In [ ]:
nombre_archivo = "meteorologia_las_palmas.csv"

df_meteorologia_raw.to_csv(
    nombre_archivo,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("Archivo generado:", nombre_archivo)
print("Registros almacenados:", len(df_meteorologia_raw))

### DESCARGAR

In [ ]:
try:
    from google.colab import files
    files.download(nombre_archivo)
except ImportError:
    print("Archivo guardado localmente:", nombre_archivo)

## 5. Calidad de los datos

Se revisan registros faltantes, duplicados, continuidad de la serie horaria y consistencia de las variables meteorológicas. Los valores de precipitación iguales a cero se consideran registros válidos; no deben confundirse con datos ausentes.

In [ ]:
df = pd.read_csv("meteorologia_las_palmas.csv", sep=";")
df["time"] = pd.to_datetime(df["time"], errors="coerce")

print("Registros analizados:", len(df))
print("Variables analizadas:", len(df.columns))
print("Datos faltantes por columna:")
display(df.isna().sum().rename("faltantes").reset_index(names="variable"))

print("Fechas inválidas:", df["time"].isna().sum())
print("Timestamps duplicados:", df["time"].duplicated().sum())

fechas_validas = df["time"].dropna().sort_values()
if not fechas_validas.empty:
    horas_esperadas = pd.date_range(fechas_validas.min(), fechas_validas.max(), freq="h")
    horas_faltantes = horas_esperadas.difference(pd.DatetimeIndex(fechas_validas))
    print("Horas faltantes dentro del periodo registrado:", len(horas_faltantes))

precipitacion = pd.to_numeric(df["precipitation"], errors="coerce")
temperatura = pd.to_numeric(df["temperature_2m"], errors="coerce")
humedad = pd.to_numeric(df["relative_humidity_2m"], errors="coerce")
print("Precipitación negativa:", (precipitacion < 0).sum())
print("Humedad fuera de 0 a 100 %:", ((humedad < 0) | (humedad > 100)).sum())
print("Valores de temperatura no numéricos:", temperatura.isna().sum())

## 6. Distribución de horas con y sin lluvia

Se considera **hora con lluvia** aquella cuyo registro horario de precipitación es mayor que cero. Los valores faltantes se mantienen como **Sin dato**, sin clasificarlos como horas secas.

In [ ]:
precipitacion = pd.to_numeric(df["precipitation"], errors="coerce")
clasificacion = pd.Series("Sin dato", index=df.index, dtype="string")
clasificacion.loc[precipitacion.notna() & (precipitacion == 0)] = "Sin lluvia"
clasificacion.loc[precipitacion.notna() & (precipitacion > 0)] = "Con lluvia"

resumen = clasificacion.value_counts(dropna=False).rename_axis("condicion").reset_index(name="horas")
display(resumen)

print("Precipitación total estimada (mm):", round(precipitacion.sum(), 2))
print("Horas con datos válidos:", precipitacion.notna().sum())

## 7. Contribución de la fuente al problema

La fuente meteorológica aporta registros horarios que pueden vincularse con la fecha y hora de los siniestros registrados en Medellín. La precipitación también permite identificar las horas sin lluvia, necesarias para comparar los accidentes en proporción al tiempo observado bajo cada condición.

La relación temporal requiere verificar que el timestamp de cada siniestro sea compatible con la zona horaria y con el periodo de acumulación de precipitación de la API. La asociación espacial dependerá de la ubicación y resolución del conjunto meteorológico seleccionado.

La comparación de accidentes por hora lluviosa o seca es descriptiva: no demuestra causalidad ni controla por sí misma el volumen de tránsito. La información de radar del proyecto es simulada y no constituye una medición histórica de velocidades en esos siniestros.